## Gauge Theory Bootstrap with varying $N_c$

## Imports

In [127]:
import os
import sys
import time
import json
from dataclasses import dataclass, field

import numpy as np
import scipy.io
import h5py

import mosek
from mosek.fusion import *
import mosek.fusion.pythonic


## Paths
setting path for importing matrices and exporting results

In [128]:
matrices_dir = os.path.join('data', 'matrices') # to import matrices
results_dir  = os.path.join('data', 'results')  # to save results
assert os.path.isdir(matrices_dir), f'Expected input dir not found: {matrices_dir}'
os.makedirs(results_dir, exist_ok=True)

# for saving previous Nc converged results as starting functional for the next Nc
results_dir_data_checkpoints=os.path.join(results_dir, 'checkpoints')
os.makedirs(results_dir_data_checkpoints, exist_ok=True)


## Config
Setting all the parameters: theoretical physical constants, discretization/truncation parameters, numerical procedure parameters
also constraints flags

In [129]:
@dataclass(frozen=True)
class TheoryParams:
    """Fixed physics constants (Nc-independent)."""
    nf: int = 2                  # number of flavors (fixed for this program)
    unit: float = 140.0          # pion mass in MeV used as the unit of energy (fixed in this program)
    fpi: float = 92.0 / 140.0    # value of fpi in pion mass unit (at Nc=3)
    alphas: float = 0.327631     # alpha_s at 2 GeV -- fixed coupling

@dataclass(frozen=True)
class DiscretizationParams:
    """Grid sizes and truncation orders for the discretized problem."""
    M: int = 100         # number of points on the physical line
    l: int = 11          # number of angular momentum per isospin
    nmax: int = 50       # truncation order for the sigma/rho1/rho2 block of `x`
    Na: int = 81         # number of coefficients per spectral-density channel
    Nb: int = 91         # number of coefficients per form-factor channel
    nu0: float = -20.0   # center of conformal map from cut-plane to unit disk
    Munphys: int = 4     # number of points in the low-energy unphysical region
    lunphys: int = 1     # number of angular momentum per isospin included in low energy constraint
    x_reg_bound: float = 1e3   # regularization bound on ||x||


@dataclass(frozen=True)
class NumProcParams:
    """Numerical control parameters for the optimization."""

    echi_0: float = 2e-3   # base value for the chiSB low-energy tolerance (echi)
    
    # sum-rule percentage error, per channel
    eS0: float = 0.05     
    eP1: float = 0.016
    eD0q: float = 3e-4

    nSR: int = 6            # number of sum rules moments for each channel
    
    # asymptotic form factor bound multipliers, per channel (bnd_X = rX * FXasym)
    rS0:  float = 8.0
    rP1:  float = 3.0
    rD0q: float = 4.0

    s0: float = (2000.0 / 140.0) ** 2   # we input UV information at s0=2GeV
    
    nWatson: int = 25       # number of Watson-iteration steps for each Nc (solve nWatson+1 times)

@dataclass(frozen=True)
class ConstraintFlags:
    """
    Explicit on/off flags for every constraint.
    """
    # normalization constraint
    ff0_S0: bool = True
    ff0_P1: bool = True
    ff0_D0q: bool = True

    # Brodsky-Lepage asymptotics constraint
    fasym_S0: bool = True
    fasym_P1: bool = True
    fasym_D0q: bool = True
    
    # spectral density IR constraint: setting leading term coefficients
    aIR_S0: bool = True
    aIR_P1: bool = True
    aIR_D0q: bool = True

    # rho asymptotics constraint
    rhoasym_S0: bool = True
    rhoasym_P1: bool = True
    rhoasym_D0q: bool = True

    # sum rule constraint
    sr_S0: bool = True
    sr_P1: bool = True
    sr_D0q: bool = True

    # SDP block for the channel
    sdp_S0: bool = True
    sdp_P1: bool = True
    sdp_D0q: bool = True

    # functional-3 term ("remaining partial waves") in compute_objective
    functional3_enabled: bool = False

    # using the previous Nc converged results to start iteration of next Nc
    use0Nc3_checkpoint: bool = True

@dataclass(frozen=True)
class RunConfig:
    theory: TheoryParams = field(default_factory=TheoryParams)
    disc: DiscretizationParams = field(default_factory=DiscretizationParams)
    num: NumProcParams = field(default_factory=NumProcParams)
    flags: ConstraintFlags = field(default_factory=ConstraintFlags)
    Nc_list: tuple = (3,4,5,6,7,8,9,10,11,12) #list of Nc

    # MOSEK feasibility tolerances
    mosek_pfeas_tol: float = 1e-6
    mosek_dfeas_tol: float = 1e-6
    mosek_relgap_tol: float = 1e-6
    mosek_mured_tol: float = 1e-8


config = RunConfig()


## Channel layout
Partial-wave sector/name mapping (sector, wave): S0→(0,0), D0→(0,1), G0→(0,2), P1→(1,0), F1→(1,1), S2→(2,0), D2→(2,1)

In [130]:
@dataclass(frozen=True)
class Block:
    start: int
    length: int

    @property
    def stop(self) -> int:
        return self.start + self.length

    def slice(self, n: int = None) -> slice:
        if n is None:
            return slice(self.start, self.stop)
        return slice(self.start, self.start + n)


class ChannelLayout:
    """
    Computes and stores all channel offset tables for a given (M, l, Na, Nb).

    partial_wave — for the "sector" matrices Mrht/Miht/Mihh. Three
    sectors of size l*M, each holding up to l channels of size M.
    offset(I, l) = sector*l*M + wave*M
    
    form_factor / spectral_density — sequential M-blocks, no sector
    structure, used for form factors (3 channels) and spectral densities (3 channels) respectively.
    
    a_vector / b_vector — sequential blocks for the optimization
    variables `a` (3 channels x Na) and `b` (3 channels x Nb).
    """

    def __init__(self, M: int, l: int, Na: int, Nb: int):
        self.M, self.l, self.Na, self.Nb = M, l, Na, Nb

        def sector_offset(sector: int, wave: int) -> int:
            return sector * l * M + wave * M

        self.partial_wave = {
            'S0': Block(sector_offset(0, 0), M),
            'D0': Block(sector_offset(0, 1), M),
            'G0': Block(sector_offset(0, 2), M),
            'P1': Block(sector_offset(1, 0), M),
            'F1': Block(sector_offset(1, 1), M),
            'S2': Block(sector_offset(2, 0), M),
            'D2': Block(sector_offset(2, 1), M),
        }

        channels = ['S0', 'P1', 'D0q']
        self.form_factor = {name: Block(i * M, M) for i, name in enumerate(channels)}
        self.spectral_density = {name: Block(i * M, M) for i, name in enumerate(channels)}
        self.a_vector = {name: Block(i * Na, Na) for i, name in enumerate(channels)}
        self.b_vector = {name: Block(i * Nb, Nb) for i, name in enumerate(channels)}


layout = ChannelLayout(config.disc.M, config.disc.l, config.disc.Na, config.disc.Nb)


## Raw matrix loading
Loads the Nc-independent matrices from `.mat` files.

In [131]:
def load_mat(matrices_dir: str, name: str) -> np.ndarray:
    path = os.path.join(matrices_dir, f'{name}.mat')
    try:
        data = scipy.io.loadmat(path)
        keys = [k for k in data.keys() if not k.startswith('__')]
        return data[keys[0]].astype(np.float64)
    except NotImplementedError:
        # v7.3 mat file (HDF5 format)
        with h5py.File(path, 'r') as f:
            keys = [k for k in f.keys()]
            return f[keys[0]][:].T.astype(np.float64)


@dataclass
class RawMatrices:
    """All Nc-independent matrices loaded directly from disk."""
    Mrht: np.ndarray      # Rehtilde
    Miht: np.ndarray      # Imhtilde
    Mihh: np.ndarray      # Imhhat
    funphy: np.ndarray    # partial waves in the unphysical region
    Acoef: np.ndarray     # amplitude at symmetric point
    Lambda3: np.ndarray   # rescaling factor for form factor in SDP
    btoImF: np.ndarray    # ReF
    btoReF: np.ndarray    # ImF
    btoF0: np.ndarray     # F(s=0)

    @classmethod
    def load(cls, matrices_dir: str) -> 'RawMatrices':
        def m(name: str) -> np.ndarray:
            return load_mat(matrices_dir, name)

        return cls(
            Mrht=m('reht'),   Miht=m('imht'),  Mihh=m('imhh'),
            funphy=m('funphys'), Acoef=m('Acoef'),
            Lambda3=m('Lambda3'),
            btoImF=m('btoImF'), btoReF=m('btoReF'), btoF0=m('btoF0'),
        )


raw = RawMatrices.load(matrices_dir)


Loads the Nc-dependent matrices from `.mat` files.

In [132]:
@dataclass
class NcMatrices:
    """Matrices that vary with Nc, loaded fresh for each Nc."""
    atorho: dict        # channel name -> matrix; keys: S0, P1, D0q
    intS0: np.ndarray   # matrices for computing sum rules
    intP1: np.ndarray
    intD0q: np.ndarray

    @classmethod
    def load(cls, matrices_dir: str, Nc: int) -> 'NcMatrices':
        def m(name: str) -> np.ndarray:
            return load_mat(matrices_dir, f'{name}_Nc{Nc:02d}')

        atorho = {ch: m(f'atorho{ch}') for ch in ('S0', 'P1', 'D0q')}
        return cls(
            atorho=atorho,
            intS0=m('intS0'), intP1=m('intP1'), intD0q=m('intD0q'),
        )

## Physical-line discretization grid
the discrete numerical points in the physical line, as well as unphysical region for imposing low energy constraints

In [133]:
@dataclass
class PhysicalGrid:
    vnu: np.ndarray           # all M points on the physical line
    n0: int                   # count of points with vnu <= s0 (SDP saturation, SR cutoff, asymptotic FF)
    vnuhigh0: np.ndarray      # vnu[n0:]  (used for spectral density asymptotics)
    vsunphys: np.ndarray      # low-energy unphysical region sample points


def build_physical_grid(disc, num) -> PhysicalGrid:
    k = np.arange(1, disc.M + 1)
    vnu = disc.nu0 + (8 - 2 * disc.nu0) / (1 + np.cos((k - 0.5) * np.pi / disc.M))

    n0 = int(np.sum(vnu <= num.s0))

    vsunphys = 0.5 + np.arange(0, disc.Munphys) / 2

    return PhysicalGrid(
        vnu=vnu, n0=n0,
        vnuhigh0=vnu[n0:],
        vsunphys=vsunphys,
    )

grid = build_physical_grid(config.disc,config.num)


## Derived matrices
Numpy preprocessing that turns the raw loaded matrices into the matrices in IR, UV or intermediate regions for imposing constraints. 

In [134]:
@dataclass
class DerivedMatrices:
    Mchi: np.ndarray

    # matrices for FF in the UV region
    btoReFS0_hi: np.ndarray
    btoImFS0_hi: np.ndarray
    btoReF_hi0: np.ndarray
    btoImF_hi0: np.ndarray

    atorho_hi: dict

    # matrices for unitarity/sdp   
    MImht: dict
    MReht: dict
    MImhh: dict
    
    btoImFh: dict

    # matrices for iterations
    MRehtS0P1D0: np.ndarray
    MImhtS0P1D0: np.ndarray
    MImhhS0P1D0: np.ndarray

    btoReF_it: np.ndarray
    btoImF_it: np.ndarray
    atorho_it: dict

    mask: np.ndarray


def build_derived_matrices(raw, disc, grid, layout, theory, nc_matrices) -> DerivedMatrices:
    M, n0 = disc.M, grid.n0

    # ─── low energy region ───
    Munphys = disc.Munphys
    lunphys = disc.lunphys
    # pw in the unphysical region
    funphyS0 = raw.funphy[0:Munphys,                                             :]                                
    funphyP1 = raw.funphy[lunphys*Munphys:lunphys*Munphys+Munphys,               :]   
    funphyS2 = raw.funphy[2*lunphys*Munphys:2*lunphys*Munphys+Munphys,           :]    

    # tree-level ratios
    R01tree = 3 * (2 * grid.vsunphys - 1) / (grid.vsunphys - 4)
    R21tree = 3 * (2 - grid.vsunphys) / (grid.vsunphys - 4)

    # deviation of S0, S2 from tree level
    MchiS0 = funphyS0 - R01tree[:, None] * funphyP1
    MchiS2 = funphyS2 - R21tree[:, None] * funphyP1
    Mchi = np.vstack([MchiS0, MchiS2])

    # ─── high energy region ───
    vnu_log = grid.vnuhigh0 / np.log(grid.vnuhigh0)       # for S0 asymptotic form factor
    vnu_lin0 = grid.vnuhigh0

    # for S0 we evaluate s*F(s)/log(s), for P1, D0 we evaluate s*F(s)
    btoReFS0_hi = vnu_log[:, None] * raw.btoReF[n0:M, :]
    btoImFS0_hi = vnu_log[:, None] * raw.btoImF[n0:M, :]
    btoReF_hi0  = vnu_lin0[:, None] * raw.btoReF[n0:M, :]
    btoImF_hi0  = vnu_lin0[:, None] * raw.btoImF[n0:M, :]

    # for rho_S0, we bound rho/s
    atorho_hi = {'S0': (1.0 / vnu_lin0[:, None]) * nc_matrices.atorho['S0'][n0:M, :]}
    for ch in ('P1', 'D0q'):
        atorho_hi[ch] = nc_matrices.atorho[ch][n0:M, :]

    # ─── SDP ───
    MImht = {ch: raw.Miht[layout.partial_wave[ch].slice()] for ch in ('S0', 'P1', 'D0')}
    MReht = {ch: raw.Mrht[layout.partial_wave[ch].slice()] for ch in ('S0', 'P1', 'D0')}
    MImhh = {ch: raw.Mihh[layout.partial_wave[ch].slice()] for ch in ('S0', 'P1', 'D0')}
    # taking S0, P1, D0 pw

    Lambda3_inv = 1.0 / raw.Lambda3.flatten()
    btoImFh = {
        ch: raw.btoImF * Lambda3_inv[layout.form_factor[ch].slice()][:, None]
        for ch in ('S0', 'P1', 'D0q')
    }
    # computing ImFhat appearing in SDP

    # ─── iteration (up to n0, i.e. up to s0) ───
    # matrices for computing pw, FF, rho that enters the iteration functional
    MRehtS0P1D0 = np.vstack([MReht[ch][0:n0, :] for ch in ('S0', 'P1', 'D0')])
    MImhtS0P1D0 = np.vstack([MImht[ch][0:n0, :] for ch in ('S0', 'P1', 'D0')])
    MImhhS0P1D0 = np.vstack([MImhh[ch][0:n0, :] for ch in ('S0', 'P1', 'D0')])

    btoReF_it = raw.btoReF[:n0, :]
    btoImF_it = raw.btoImF[:n0, :]
    atorho_it = {ch: nc_matrices.atorho[ch][:n0, :] for ch in ('S0', 'P1', 'D0q')}
    
    #  — select F1, S2, D2 partial waves up to n0
    n_cone = raw.Mrht.shape[0]
    mask = np.zeros(n_cone, dtype=bool)
    for ch in ('F1', 'S2', 'D2'):
        mask[layout.partial_wave[ch].slice(n0)] = True

    return DerivedMatrices(
        Mchi=Mchi,
        btoReFS0_hi=btoReFS0_hi, btoImFS0_hi=btoImFS0_hi,
        btoReF_hi0=btoReF_hi0, btoImF_hi0=btoImF_hi0,
        atorho_hi=atorho_hi,
        MImht=MImht, MReht=MReht, MImhh=MImhh,
        btoImFh=btoImFh,
        MRehtS0P1D0=MRehtS0P1D0, MImhtS0P1D0=MImhtS0P1D0, MImhhS0P1D0=MImhhS0P1D0,
        btoReF_it=btoReF_it, btoImF_it=btoImF_it,
        atorho_it=atorho_it,
        mask=mask,
    )


## $N_c$-dependent parameters
Everything that depends on the number of colors `Nc`: pQCD/Brodsky-Lepage asymptotic constants, the sum-rule data (`SVZSR_Nc##.csv`), and the derived sum-rule errors, asymptotic form-factor bounds (`bnd_*`), and asymptotic spectral-density bounds (`rhobd_*`).

In [135]:
@dataclass
class NcParams:
    Nc: int
    CF: float
    echi: float           # low energy matching tolerance
    qcdfesr: np.ndarray   # qcd sum rules

    # Brodsky-Lepage asymptotic form factors
    FS0asym: float
    FP1asym: float
    FD0qasym: float

    # pQCD LO asymptotic spectral densities (rhohat/s for S0)
    rhoS0asym: float
    rhoP1asym: float
    rhoD0qasym: float

    # sum-rule errors, per channel
    errS0: np.ndarray
    errP1: np.ndarray
    errD0q: np.ndarray

    # asymptotic form-factor bounds: bnd_X = rX * FXasym
    bnd_S0: float
    bnd_P1: float
    bnd_D0q: float

    # asymptotic spectral-density bounds, evaluated on vnuhigh0 (high energy points)
    rhobd_S0: np.ndarray
    rhobd_P1: np.ndarray
    rhobd_D0q: np.ndarray


def build_nc_params(Nc, theory, disc, num, grid, matrices_dir, echi_per_Nc=True) -> NcParams:
    CF = (Nc**2 - 1) / (2 * Nc)
    fpi = theory.fpi

    # asymptotic form factor from Brodsky-Lepage computation
    FS0asym  = 1.0
    FP1asym  = 72  * np.pi * theory.alphas * fpi**2 * (Nc**2 - 1) / (4 * Nc**2)
    FD0qasym = 84  * np.pi * theory.alphas * fpi**2 * (Nc**2 - 1) / (2 * Nc**2)

    # asymptotic value of rhohat from pQCD (leading)
    rhoS0asym   = 0.00375578 * Nc    # rhohat/s
    rhoP1asym   = 2 * Nc
    rhoD0qasym  = 4 * Nc

    # echi for low energy constraints -- Nc dependent or not
    echi = num.echi_0 * 3 / Nc if echi_per_Nc else num.echi_0

    # qcd sum rules
    qcd_path = os.path.join(matrices_dir, f'SVZSR_Nc{Nc:02d}.csv')
    qcdfesr = np.loadtxt(qcd_path)

    # ── sum-rule error (fixed percentage of the QCD sum-rule value) ──
    nSR = num.nSR
    errS0   = num.eS0   * qcdfesr[0:nSR]
    errP1   = num.eP1   * qcdfesr[nSR:2*nSR]
    errD0q  = num.eD0q  * qcdfesr[2*nSR:2*nSR+nSR]

    # ── asymptotic form-factor bounds, bnd_X = rX * FXasym ──
    bnd_S0  = float(num.rS0  * FS0asym)
    bnd_P1  = float(num.rP1  * FP1asym)
    bnd_D0q = float(num.rD0q * FD0qasym)   

    # ── asymptotic spectral-density bounds, evaluated on vnuhigh0 ──
    vhigh = grid.vnuhigh0
    # we choose a form that goes to 0 at infinity and 1 at s0
    rhobd_S0   = (num.s0 / (2*vhigh - num.s0))     * rhoS0asym
    rhobd_P1   = (num.s0 / (2*vhigh - num.s0))     * rhoP1asym
    rhobd_D0q  = (num.s0 / (2*vhigh - num.s0))     * rhoD0qasym

    return NcParams(
        Nc=Nc, CF=CF, echi=echi, qcdfesr=qcdfesr,
        FS0asym=FS0asym, FP1asym=FP1asym, FD0qasym=FD0qasym,
        rhoS0asym=rhoS0asym, rhoP1asym=rhoP1asym, rhoD0qasym=rhoD0qasym,
        errS0=errS0, errP1=errP1, errD0q=errD0q,
        bnd_S0=bnd_S0, bnd_P1=bnd_P1, bnd_D0q=bnd_D0q,
        rhobd_S0=rhobd_S0, rhobd_P1=rhobd_P1, rhobd_D0q=rhobd_D0q,
    )


## SDP block helpers
Build the 3x3-block SDP constraint matrices out of the partial-wave, form-factor, and spectral-density expressions for a channel (or pair of channels), the 3x3 complex SDP is equivalent to a 6x6 real SDP

In [136]:
def sdp_3(M_, reht, imht, imhh, ref, imf, imfh, rho, name):
    n = reht.shape[0]
    onef  = Expr.ones(n)
    zerof = Expr.zeros(n)
    imhh2 = 2.0 * imhh
    imfh2 = 2.0 * imfh
    block = Expr.stack(2, [
        Expr.hstack([onef,   reht,    ref,     zerof,    -imht,    -imf]),
        Expr.hstack([reht,   imhh2,   imfh2,   imht,     zerof,    zerof]),
        Expr.hstack([ref,    imfh2,   rho,     imf,      zerof,    zerof]),
        Expr.hstack([zerof,  imht,    imf,     onef,     reht,     ref]),
        Expr.hstack([-imht,  zerof,   zerof,   reht,     imhh2,    imfh2]),
        Expr.hstack([-imf,   zerof,   zerof,   ref,      imfh2,    rho]),
    ])
    M_.constraint(name, block, Domain.inPSDCone(6, n))


## Model builder (`gtb_build`)
Builds the MOSEK Fusion model: variables, unitarity/regularization constraints, form-factor normalization/asymptotics, spectral-density IR/UV/asymptotics/sum-rule constraints, SDP blocks, and the (initially zero) objective.

In [137]:
@dataclass
class GtbModel:
    model: Model
    obj_param: Parameter
    x: Variable     # amplitude parameters
    a: Variable     # spectral density parameters
    b: Variable     # form factor parameters
    x_dim: int
    a_dim: int
    b_dim: int


def build_model(config, layout, grid, raw, derived, nc_params, nc_matrices) -> GtbModel:
    disc, num, flags, theory = config.disc, config.num, config.flags, config.theory
    nf = theory.nf # number of flavor fixed
    CF = nc_params.CF

    M_ = Model('gtb')
    M_.setLogHandler(sys.stdout)

    M_.setSolverParam('intpntCoTolPfeas',  config.mosek_pfeas_tol)
    M_.setSolverParam('intpntCoTolDfeas',  config.mosek_dfeas_tol)
    M_.setSolverParam('intpntCoTolRelGap', config.mosek_relgap_tol)
    M_.setSolverParam('intpntCoTolMuRed',  config.mosek_mured_tol)

    # ──────────────────────────  variables declaration  ────────────────────────
    nmax, Na, Nb = disc.nmax, disc.Na, disc.Nb
    x_dim    = 1 + 2*nmax + nmax*nmax + (nmax+1)*nmax // 2
    a_dim    = 3 * Na
    b_dim    = 3 * Nb
    full_dim = x_dim + a_dim + b_dim
    x = M_.variable('x', x_dim, Domain.unbounded())
    a = M_.variable('a', a_dim, Domain.unbounded())
    b = M_.variable('b', b_dim, Domain.unbounded())

    # ──────────────────────────  variables slicing  ────────────────────────
    def a_slice(ch):
        blk = layout.a_vector[ch]
        return a.slice(blk.start, blk.stop)

    def b_slice(ch):
        blk = layout.b_vector[ch]
        return b.slice(blk.start, blk.stop)

    bS0, bP1, bD0q = (b_slice(ch) for ch in ('S0', 'P1', 'D0q'))
    aS0, aP1, aD0q = (a_slice(ch) for ch in ('S0', 'P1', 'D0q'))

    # ──────────────────────────  amplitudes  ────────────────────────
    # regularization
    M_.constraint('reg', Expr.vstack(Expr.constTerm(disc.x_reg_bound), x), Domain.inQCone())

    # quadratic cone unitarity
    Reht = raw.Mrht @ x; Imht = raw.Miht @ x; Imhh = raw.Mihh @ x
    n_cone = raw.Mrht.shape[0]
    ones = np.ones(n_cone)
    cone_mat = Expr.hstack([Imhh, Expr.constTerm(ones), Reht, Imht])
    M_.constraint('unitarity', cone_mat, Domain.inRotatedQCone())

    # low energy unphysical region
    chi_vec = derived.Mchi @ x
    M_.constraint('chiSB', Expr.vstack([Expr.constTerm(nc_params.echi), chi_vec]), Domain.inQCone())

    # ──────────────────────────  form factors  ────────────────────────
    # form factor normalization
    if flags.ff0_S0:
        M_.constraint('FF0_S0', Expr.dot(raw.btoF0.flatten(), bS0), Domain.equalsTo(1.0))
    if flags.ff0_P1:
        M_.constraint('FF0_P1', Expr.dot(raw.btoF0.flatten(), bP1), Domain.equalsTo(1.0))
    # the normalization of T_q is approximate from operator defined at infinity, we include a small tolerance for this normalization
    if flags.ff0_D0q:
        M_.constraint('FF0_D0q', Expr.dot(raw.btoF0.flatten(), bD0q), Domain.inRange(nf/(4*CF + nf) - nf*0.1/(4*CF + nf), nf/(4*CF + nf) + nf*0.1 /(4*CF + nf)))

    # Brodsky-Lepage asymptotics
    ReFF_S0_hi  = derived.btoReFS0_hi @ bS0;  ImFF_S0_hi  = derived.btoImFS0_hi @ bS0
    ReFF_P1_hi  = derived.btoReF_hi0 @ bP1;   ImFF_P1_hi  = derived.btoImF_hi0 @ bP1
    ReFF_D0q_hi = derived.btoReF_hi0 @ bD0q;  ImFF_D0q_hi = derived.btoImF_hi0 @ bD0q
    n_hi0 = len(grid.vnuhigh0) # number of poitns in the high energy region
    
    if flags.fasym_S0:
        M_.constraint('Fasym_S0', Expr.hstack([Expr.constTerm(np.full(n_hi0, nc_params.bnd_S0)), ReFF_S0_hi, ImFF_S0_hi]), Domain.inQCone())
    if flags.fasym_P1:
        M_.constraint('Fasym_P1', Expr.hstack([Expr.constTerm(np.full(n_hi0, nc_params.bnd_P1)), ReFF_P1_hi, ImFF_P1_hi]), Domain.inQCone())
    if flags.fasym_D0q:
        M_.constraint('Fasym_D0q', Expr.hstack([Expr.constTerm(np.full(n_hi0, nc_params.bnd_D0q)), ReFF_D0q_hi, ImFF_D0q_hi]), Domain.inQCone())

    # ──────────────────────────  spectral density  ────────────────────────
    # aIR coefficients
    if flags.aIR_S0:
        M_.constraint('aS0_IR', aS0.index(0), Domain.inRange(1, 1))
    if flags.aIR_P1:
        M_.constraint('aP1_IR', aP1.index(0), Domain.inRange(1, 1))
    if flags.aIR_D0q:
        M_.constraint('aD0q_IR', aD0q.index(0), Domain.inRange(1, 1))

    # spectral density at high energies (above s0)
    rho_S0_hi   = derived.atorho_hi['S0']   @ aS0
    rho_P1_hi   = derived.atorho_hi['P1']   @ aP1
    rho_D0q_hi  = derived.atorho_hi['D0q']  @ aD0q
    if flags.rhoasym_S0:
        M_.constraint('rhoasym_S0', rho_S0_hi, Domain.inRange(nc_params.rhoS0asym - nc_params.rhobd_S0, nc_params.rhoS0asym + nc_params.rhobd_S0))
    if flags.rhoasym_P1:
        M_.constraint('rhoasym_P1', rho_P1_hi, Domain.inRange(nc_params.rhoP1asym - nc_params.rhobd_P1, nc_params.rhoP1asym + nc_params.rhobd_P1))
    if flags.rhoasym_D0q:
        M_.constraint('rhoasym_D0q', rho_D0q_hi, Domain.inRange(nc_params.rhoD0qasym - nc_params.rhobd_D0q, nc_params.rhoD0qasym + nc_params.rhobd_D0q))

    # sum rules with errors
    nSR = num.nSR
    qcdfesr = nc_params.qcdfesr
    if flags.sr_S0:
        M_.constraint('srS0', nc_matrices.intS0 @ aS0, Domain.inRange(qcdfesr[0:nSR] - nc_params.errS0, qcdfesr[0:nSR] + nc_params.errS0))
    if flags.sr_P1:
        M_.constraint('srP1', nc_matrices.intP1 @ aP1, Domain.inRange(qcdfesr[nSR:2*nSR] - nc_params.errP1, qcdfesr[nSR:2*nSR] + nc_params.errP1))
    if flags.sr_D0q:
        M_.constraint('srD0q', nc_matrices.intD0q[:nSR] @ aD0q, Domain.inRange(qcdfesr[2*nSR:2*nSR+nSR] - nc_params.errD0q, qcdfesr[2*nSR:2*nSR+nSR] + nc_params.errD0q))

    # ──────────────────────────  sdp ────────────────────────
    # partial waves in S0, P1, D0
    RehtS0 = derived.MReht['S0'] @ x; ImhtS0 = derived.MImht['S0'] @ x; ImhhS0 = derived.MImhh['S0'] @ x
    RehtP1 = derived.MReht['P1'] @ x; ImhtP1 = derived.MImht['P1'] @ x; ImhhP1 = derived.MImhh['P1'] @ x
    RehtD0 = derived.MReht['D0'] @ x; ImhtD0 = derived.MImht['D0'] @ x; ImhhD0 = derived.MImhh['D0'] @ x

    # form factors in S0, P1, D0
    ReFFS0  = raw.btoReF @ bS0;  ImFFS0  = raw.btoImF @ bS0;  ImFFhS0  = derived.btoImFh['S0']  @ bS0
    ReFFP1  = raw.btoReF @ bP1;  ImFFP1  = raw.btoImF @ bP1;  ImFFhP1  = derived.btoImFh['P1']  @ bP1
    ReFFD0q = raw.btoReF @ bD0q; ImFFD0q = raw.btoImF @ bD0q; ImFFhD0q = derived.btoImFh['D0q'] @ bD0q

    # spectral densities in S0, P1, D0
    rhoS0   = nc_matrices.atorho['S0']   @ aS0
    rhoP1   = nc_matrices.atorho['P1']   @ aP1
    rhoD0q  = nc_matrices.atorho['D0q']  @ aD0q

    # sdp for S0, P1, D0
    if flags.sdp_S0:
        sdp_3(M_, RehtS0, ImhtS0, ImhhS0, ReFFS0, ImFFS0, ImFFhS0, rhoS0, "sdp_S0")
    if flags.sdp_P1:
        sdp_3(M_, RehtP1, ImhtP1, ImhhP1, ReFFP1, ImFFP1, ImFFhP1, rhoP1, "sdp_P1")
    if flags.sdp_D0q:
        sdp_3(M_, RehtD0, ImhtD0, ImhhD0, ReFFD0q, ImFFD0q, ImFFhD0q, rhoD0q, "sdp_D0q")

    # ──────────────────────────  objective ────────────────────────
    obj_param = M_.parameter('obj', full_dim)
    obj_param.setValue(np.zeros(full_dim))   # first solve is feasibility (at Nc=3)
    M_.objective(ObjectiveSense.Maximize, Expr.dot(obj_param, Expr.vstack([x, a, b])))

    return GtbModel(model=M_, obj_param=obj_param, x=x, a=a, b=b,
                     x_dim=x_dim, a_dim=a_dim, b_dim=b_dim)


## Objective update (`compute_objective`)
Builds the linear objective vector `c = (cx, ca, cb)` for the *next* solve, from the *previous* solve's results (`diag`). This is the Watson-iteration fixed-point scheme: each solve's result feeds into the next solve's objective.

In [138]:
def compute_objective(diag, config, layout, grid, raw, derived):
    disc, flags = config.disc, config.flags
    n0 = grid.n0
    Na, Nb, M = disc.Na, disc.Nb, disc.M
    x_dim = 1 + 2*disc.nmax + disc.nmax*disc.nmax + (disc.nmax+1)*disc.nmax // 2
    a_dim = 3 * Na
    b_dim = 3 * Nb

    cx = np.zeros(x_dim); ca = np.zeros(a_dim); cb = np.zeros(b_dim)

    # ── functional 1: Watson partial-wave & form factor phase aligning ──────────────
    htS0P1D0_prev = diag['htS0P1D0']    # complex (3*n0,)
    FF3_prev      = diag['FF3']         # complex (3*n0,) [S0,P1,D0q]
    FF3_S0P1D0    = FF3_prev[0:3*n0]
    ImFF3h_prev   = diag['ImFFh']
    ImFF3h_S0P1D0 = ImFF3h_prev[0:3*n0]

    htS0P1D0wat   = np.abs(htS0P1D0_prev) * FF3_S0P1D0 / np.abs(FF3_S0P1D0)
    # using the phase of form factor to set the phase of partial waves
    watson_Re     = np.real(htS0P1D0wat)
    watson_Im     = np.imag(htS0P1D0wat)
    cx += watson_Im @ derived.MImhtS0P1D0
    cx += watson_Re @ derived.MRehtS0P1D0
    cx -= np.ones(3*n0) @ derived.MImhhS0P1D0

    # ── functional 2: form factor spectral density saturation ────────────────────
    ReFF_prev = diag['ReFF'];        ImFF_prev = diag['ImFF']
    wReFS0  = ReFF_prev[0:n0];       wImFS0  = ImFF_prev[0:n0]
    wReFP1  = ReFF_prev[M:M+n0];     wImFP1  = ImFF_prev[M:M+n0]
    wReFD0q = ReFF_prev[2*M:2*M+n0]; wImFD0q = ImFF_prev[2*M:2*M+n0]

    b_S0, b_P1, b_D0q = layout.b_vector['S0'], layout.b_vector['P1'], layout.b_vector['D0q']
    a_S0, a_P1, a_D0q = layout.a_vector['S0'], layout.a_vector['P1'], layout.a_vector['D0q']

    cb[b_S0.slice()]  += 2*(wReFS0  @ derived.btoReF_it + wImFS0  @ derived.btoImF_it)
    cb[b_P1.slice()]  += 2*(wReFP1  @ derived.btoReF_it + wImFP1  @ derived.btoImF_it)
    cb[b_D0q.slice()] += 2*(wReFD0q @ derived.btoReF_it + wImFD0q @ derived.btoImF_it)
    ca[a_S0.slice()]  -= np.ones(n0) @ derived.atorho_it['S0']
    ca[a_P1.slice()]  -= np.ones(n0) @ derived.atorho_it['P1']
    ca[a_D0q.slice()] -= np.ones(n0) @ derived.atorho_it['D0q']

    # ── functional 3: remaining partial waves ──────
    if flags.functional3_enabled:
        Reht_prev = diag['Reht']; Imht_prev = diag['Imht']
        mask = derived.mask
        wRe_rest = Reht_prev[mask]
        wIm_rest = Imht_prev[mask]
        cx += wRe_rest @ raw.Mrht[mask, :] + wIm_rest @ raw.Miht[mask, :]
        cx -= np.ones(mask.sum()) @ raw.Mihh[mask, :]

    return cx, ca, cb


## Solve step (`gtb_solve`)
Updates the objective from the previous iteration's diagnostics (except for 0th iteration), solves, checks feasibility, and extracts both the solution (`sol`) and the diagnostics needed for the *next* `compute_objective` call (`diag`).

In [139]:
def solve_iteration(gtb_model, diag, ni, config, layout, grid, raw, derived, nc_matrices):
    M_ = gtb_model.model
    disc = config.disc
    n0, l, M = grid.n0, disc.l, disc.M
    Na, Nb = disc.Na, disc.Nb

    # ── update objective ───────────────────────────────────────────────────
    if ni > 0:
        cx, ca, cb = compute_objective(diag, config, layout, grid, raw, derived)
        c = np.concatenate([cx, ca, cb])
        gtb_model.obj_param.setValue(c)

    # ── solve ──────────────────────────────────────────────────────────────
    M_.setSolverParam('intpntSolveForm', 'free')
    solve_start = time.perf_counter()
    M_.solve()
    solve_time = time.perf_counter() - solve_start
    print(f'Solve time: {solve_time:.2f} sec')

    status = M_.getProblemStatus(SolutionType.Default)
    status_str = str(status)
    if status not in [ProblemStatus.PrimalAndDualFeasible, ProblemStatus.PrimalFeasible]:
        raise RuntimeError(f'Solver failed with status: {status_str}')

    # ──────────────────────────  extract results  ────────────────────────
    x_sol = np.array(gtb_model.x.level())
    a_sol = np.array(gtb_model.a.level())
    b_sol = np.array(gtb_model.b.level())

    def a_part(ch):
        blk = layout.a_vector[ch]
        return a_sol[blk.slice()]

    def b_part(ch):
        blk = layout.b_vector[ch]
        return b_sol[blk.slice()]

    Asym = float(raw.Acoef.flatten() @ x_sol)
    lambda_ = (3 * np.pi * Asym) / 4

    Reht_sol = raw.Mrht @ x_sol
    Imht_sol = raw.Miht @ x_sol

    channels = ['S0', 'P1', 'D0q']
    ReFF_sol = np.concatenate([raw.btoReF @ b_part(ch) for ch in channels]) # form factors
    ImFF_sol = np.concatenate([raw.btoImF @ b_part(ch) for ch in channels])
    ImFFh3_sol = np.concatenate([derived.btoImFh[ch] @ b_part(ch) for ch in channels])

    spec3 = np.concatenate([nc_matrices.atorho[ch] @ a_part(ch) for ch in channels]) # spectral density

    # partial waves for Watson (S0, P1, D0 sectors, using the partial-wave layout)
    S0_blk, P1_blk, D0_blk = layout.partial_wave['S0'], layout.partial_wave['P1'], layout.partial_wave['D0']
    RehtS0_s = Reht_sol[S0_blk.slice(n0)]; ImhtS0_s = Imht_sol[S0_blk.slice(n0)]
    RehtP1_s = Reht_sol[P1_blk.slice(n0)]; ImhtP1_s = Imht_sol[P1_blk.slice(n0)]
    RehtD0_s = Reht_sol[D0_blk.slice(n0)]; ImhtD0_s = Imht_sol[D0_blk.slice(n0)]

    htS0P1D0_sol = (np.concatenate([RehtS0_s, RehtP1_s, RehtD0_s]) +
                    1j * np.concatenate([ImhtS0_s, ImhtP1_s, ImhtD0_s]))

    ff_blk = layout.form_factor
    FF3_sol = np.concatenate([
        ReFF_sol[ff_blk[ch].slice(n0)] + 1j * ImFF_sol[ff_blk[ch].slice(n0)]
        for ch in channels
    ])

    sol = {
        'lambda': lambda_,
        'status': status_str,
        'sigrho': x_sol,
        'a': a_sol,
        'b': b_sol,
        'Reht': Reht_sol,
        'Imht': Imht_sol,
        'ReFF': ReFF_sol,
        'ImFF': ImFF_sol,
        'spec3': spec3,
    }

    diag = {
        'htS0P1D0': htS0P1D0_sol,   # complex (3*n0,)
        'FF3': FF3_sol,              # complex (3*n0,)
        'ReFF': ReFF_sol,            # for functional 2
        'ImFF': ImFF_sol,
        'ImFFh': ImFFh3_sol,
        'Reht': Reht_sol,            # for functional 3
        'Imht': Imht_sol,
    }

    return sol, diag


## Result serialization

In [140]:
def serialize_solution(sol, layout) -> dict:
    result = {'lambda': sol['lambda']}

    for ch in ('S0', 'D0', 'G0', 'P1', 'F1'):
        blk = layout.partial_wave[ch]
        result[f'Reht{ch}'] = sol['Reht'][blk.slice()].tolist()
        result[f'Imht{ch}'] = sol['Imht'][blk.slice()].tolist()

    for ch in ('S0', 'P1', 'D0q'):
        blk = layout.spectral_density[ch]
        result[f'spec{ch}'] = sol['spec3'][blk.slice()].tolist()

    for ch in ('S0', 'P1', 'D0q'):
        blk = layout.form_factor[ch]
        result[f'ReFF{ch}'] = sol['ReFF'][blk.slice()].tolist()
        result[f'ImFF{ch}'] = sol['ImFF'][blk.slice()].tolist()

    result['sigrho'] = sol['sigrho'].tolist()

    for ch in ('S0', 'P1', 'D0q'):
        blk = layout.a_vector[ch]
        result[f'a{ch}'] = sol['a'][blk.slice()].tolist()

    for ch in ('S0', 'P1', 'D0q'):
        blk = layout.b_vector[ch]
        result[f'b{ch}'] = sol['b'][blk.slice()].tolist()

    return result


## Run loop ($N_c$ x Watson iterations)
For each `Nc`: build Nc-dependent params and matrices, build a fresh model, then solve `nWatson + 1` times, each time refining the objective from the previous solve (`compute_objective`/`solve_iteration`), saving each feasible iteration's solution to JSON.

In [141]:
for Nc in config.Nc_list:
    nc_params = build_nc_params(Nc, config.theory, config.disc, config.num, grid, matrices_dir)
    nc_matrices = NcMatrices.load(matrices_dir, Nc)
    derived = build_derived_matrices(raw, config.disc, grid, layout, config.theory, nc_matrices)

    build_start = time.perf_counter()
    gtb_model = build_model(config, layout, grid, raw, derived, nc_params, nc_matrices)
    print(f'Model built in {time.perf_counter() - build_start:.2f} sec')
    checkpoint_path = os.path.join(results_dir_data_checkpoints, f'gtb_qcd_logs_Nc{Nc:02d}_watson_checkpoint.npz')
    
    try:
        diag = {}
        start_ni = 0
        feasible = True
        if os.path.exists(checkpoint_path):
                    checkpoint = np.load(checkpoint_path, allow_pickle=False)
                    try:
                        last_ni = int(checkpoint['last_ni'])
                        diag = {
                            'htS0P1D0': checkpoint['htS0P1D0'],
                            'FF3':      checkpoint['FF3'],
                            'ReFF':     checkpoint['ReFF'],
                            'ImFF':     checkpoint['ImFF'],
                            'ImFFh':    checkpoint['ImFFh'],
                            'Reht':     checkpoint['Reht'],
                            'Imht':     checkpoint['Imht'],
                        }
                    finally:
                        checkpoint.close()
        
                    start_ni = last_ni + 1
                    if start_ni <= config.num.nWatson:
                        print(f'Resuming Nc={Nc} from saved Watson checkpoint Witer={last_ni}; next Witer={start_ni}.')
                    else:
                        print(f'Nc={Nc} already has checkpoint through Witer={last_ni}; skipping.')
                        continue
        if start_ni==0 and Nc>3 and config.flags.use0Nc3_checkpoint:  # for Nc>3, use Nc=3 resutls to initiate iteration
                    checkpoint_pathNc3 = os.path.join(results_dir_data_checkpoints, f'gtb_qcd_logs_Nc{3:02d}_watson_checkpoint.npz')
                    if os.path.exists(checkpoint_pathNc3):
                        checkpoint = np.load(checkpoint_pathNc3, allow_pickle=False)
                        try:
                          last_ni =0
                          diag = {
                            'htS0P1D0': checkpoint['htS0P1D0'],
                            'FF3':      checkpoint['FF3'],
                            'ReFF':     checkpoint['ReFF'],
                            'ImFF':     checkpoint['ImFF'],
                            'ImFFh':    checkpoint['ImFFh'],
                            'Reht':     checkpoint['Reht'],
                            'Imht':     checkpoint['Imht'],
                            }
                        finally:
                            checkpoint.close()
        
                        start_ni = last_ni + 1
                        if start_ni <= config.num.nWatson:
                            print(f'Resuming Nc={Nc} from saved Watson checkpoint Witer={last_ni}; next Witer={start_ni}.')
                        else:
                            print(f'Nc={Nc} already has checkpoint through Witer={last_ni}; skipping.')
                            continue
                       
        print(f"About to start Watson iterations for Nc={Nc}, start_ni={start_ni}")
        for ni in range(start_ni,config.num.nWatson + 1):
            iter_start = time.perf_counter()
            sol, diag = solve_iteration(gtb_model, diag, ni, config, layout, grid, raw, derived, nc_matrices)

            elapsed = (time.perf_counter() - iter_start) / 60
            print()
            print(f'Nc={Nc}, Witer={ni}, lambda={sol["lambda"]:.6f}, '
                  f'status: {sol["status"]}, time: {elapsed:.2f} min')

#            if sol['status'] != str(ProblemStatus.PrimalAndDualFeasible):
#                print(f'Nc={Nc} did not reach PrimalAndDualFeasible at iter {ni}, skipping.')
#                feasible = False
#                break

            result = serialize_solution(sol, layout)
            out_path = os.path.join(
                results_dir, f'gtb_qcd_Nc{Nc:02d}_W{ni:02d}.json'
            )
            with open(out_path, 'w') as f:
                json.dump(result, f)

            # -- Save Watson resume checkpoint after the result file is durable --
            checkpoint_tmp = checkpoint_path + '.tmp.npz'
            np.savez_compressed(
                            checkpoint_tmp,
                            last_ni=np.array(ni, dtype=np.int64),
                            htS0P1D0=diag['htS0P1D0'],
                            FF3=diag['FF3'],
                            ReFF=diag['ReFF'],
                            ImFF=diag['ImFF'],
                            ImFFh=diag['ImFFh'],
                            Reht=diag['Reht'],
                            Imht=diag['Imht'],
                        )
            os.replace(checkpoint_tmp, checkpoint_path)


        if not feasible:
            print(f'Nc={Nc} skipped — no feasible solution found.')
            continue

    finally:
        gtb_model.model.dispose()


MOSEK warning 705 (MSK_RES_WRN_ZEROS_IN_SPARSE_ROW): 59 (nearly) zero elements are specified in sparse row ''(0) of matrix 'A'.
MOSEK warning 705 (MSK_RES_WRN_ZEROS_IN_SPARSE_ROW): 59 (nearly) zero elements are specified in sparse row ''(1) of matrix 'A'.
MOSEK warning 705 (MSK_RES_WRN_ZEROS_IN_SPARSE_ROW): 59 (nearly) zero elements are specified in sparse row ''(2) of matrix 'A'.
Model built in 5.89 sec
About to start Watson iterations for Nc=3, start_ni=0
Problem
  Name                   : gtb             
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 87              
  Affine conic cons.     : 3665 (23575 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 4393            
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started